In [2]:
# ==============================================================================
# CELL: READ DATA & REPRODUCE ABSORBER STATISTICS TABLE
# ==============================================================================
import pandas as pd
import numpy as np

In [10]:
# --- 1. CONFIGURAÇÃO DO ARQUIVO ---
# Troque SIM_NAME pelo nome exato do arquivo gerado (ex: 'LH_1_Random_10000Sk_Combined.pkl')
file_path = 'fiducial_Grid_2883Sk.pkl' 

print("Loading the file...")
df = pd.read_pickle(file_path)

# --- 2. INSPEÇÃO DA ESTRUTURA ---
print("\n" + "="*50)
print(" ESTRUTURA DO DATAFRAME (Primeiras Linhas)")
print("="*50)
display(df.head())

print("\n" + "="*50)
print(" INFORMAÇÕES DAS COLUNAS")
print("="*50)
df.info()

Loading the file...

 ESTRUTURA DO DATAFRAME (Primeiras Linhas)


,skewer_idx,axis,pos_x_code,pos_y_code,pos_z_code,chunk_id,log_n_hi,total_col_den,flux,n_hi_col,temp_k,wave,vel_axis,chi_axis,is_zero_col_den
0,0,1,0.001,8.064516,8.064516,0,16.788746,6.148170e+16,"[0.8739733, 0.87361586, 0.8736158, 0.8739993, ...","[1253323300000.0, 1253323300000.0, 10688387000...","[1598480.1, 1598480.1, 1862122.9, 1974536.9, 1...","[1215.67, 1215.7106, 1215.7511, 1215.7916, 121...","[0.0, 10.0008135, 20.001627, 30.002441, 40.003...","[0.0, 0.088652484, 0.17730497, 0.26595744, 0.3...",False
1,1,1,0.001,8.064516,24.193548,0,16.787305,6.127806e+16,"[0.13923666, 0.08793253, 0.1626225, 0.2327051,...","[14399720000000.0, 14399720000000.0, 143997200...","[38904.734, 38904.734, 38904.734, 38904.734, 3...","[1215.67, 1215.7106, 1215.7511, 1215.7916, 121...","[0.0, 10.0008135, 20.001627, 30.002441, 40.003...","[0.0, 0.088652484, 0.17730497, 0.26595744, 0.3...",False
2,2,1,0.001,8.064516,40.322581,0,16.766498,5.841145e+16,"[0.95960665, 0.7376732, 0.32867196, 0.13197722...","[8624045000000.0, 8624045000000.0, 86240450000...","[139159.75, 139159.75, 139159.75, 139159.75, 1...","[1215.67, 1215.7106, 1215.7511, 1215.7916, 121...","[0.0, 10.0008135, 20.001627, 30.002441, 40.003...","[0.0, 0.088652484, 0.17730497, 0.26595744, 0.3...",False
3,3,1,0.001,8.064516,56.451613,0,16.809286,6.445943e+16,"[0.63953316, 0.6485302, 0.64677674, 0.6383064,...","[3277522700000.0, 3277522700000.0, 32775227000...","[418845.5, 418845.5, 418845.5, 418845.5, 41884...","[1215.67, 1215.7106, 1215.7511, 1215.7916, 121...","[0.0, 10.0008135, 20.001627, 30.002441, 40.003...","[0.0, 0.088652484, 0.17730497, 0.26595744, 0.3...",False
4,4,1,0.001,8.064516,72.580645,0,16.798258,6.284323e+16,"[0.992653, 0.9721776, 0.9250479, 0.8543956, 0....","[2494270300000.0, 2494270300000.0, 24942703000...","[262473.28, 262473.28, 262473.28, 262473.28, 2...","[1215.67, 1215.7106, 1215.7511, 1215.7916, 121...","[0.0, 10.0008135, 20.001627, 30.002441, 40.003...","[0.0, 0.088652484, 0.17730497, 0.26595744, 0.3...",False



 INFORMAÇÕES DAS COLUNAS
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2883 entries, 0 to 2882
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   skewer_idx       2883 non-null   int64  
 1   axis             2883 non-null   int64  
 2   pos_x_code       2883 non-null   float64
 3   pos_y_code       2883 non-null   float64
 4   pos_z_code       2883 non-null   float64
 5   chunk_id         2883 non-null   int64  
 6   log_n_hi         2883 non-null   float64
 7   total_col_den    2883 non-null   float64
 8   flux             2883 non-null   object 
 9   n_hi_col         2883 non-null   object 
 10  temp_k           2883 non-null   object 
 11  wave             2883 non-null   object 
 12  vel_axis         2883 non-null   object 
 13  chi_axis         2883 non-null   object 
 14  is_zero_col_den  2883 non-null   bool   
dtypes: bool(1), float64(5), int64(3), object(6)
memory usage: 318.3+ KB


In [11]:
# --- 3. CÁLCULO DA DENSIDADE TOTAL ---
# A coluna total do skewer é a soma do gás denso (DLA) e difuso (No_DLA)
df['total_N_HI_skewer'] = df['total_col_den']

# --- 4. CLASSIFICAÇÃO DOS SKEWERS (BINAGEM) ---
# Definindo os limites exatos fornecidos pela imagem (em escala linear)
# N_HI em [atoms cm^-2]
bins = [0, 1.6e17, 1e19, 2e20, 1e21, np.inf]
labels = ['Lyman-α forest', 'LLS', 'Sub-DLA', 'Small DLA', 'Large DLA']

# right=False garante que o intervalo seja [min, max), ou seja, inclui o mínimo e exclui o máximo, 
# assim como N(HI)_min <= N < N(HI)_max
df['category'] = pd.cut(df['total_N_HI_skewer'], bins=bins, labels=labels, right=False)

# --- 5. CÁLCULO DAS PORCENTAGENS ---
total_skewers = len(df)
# Conta quantos skewers caíram em cada categoria preservando a ordem dos labels
category_counts = df['category'].value_counts(sort=False) 
category_percentages = (category_counts / total_skewers) * 100



In [12]:
how_many_zeros = np.array(np.where(df['total_N_HI_skewer'] == 0))
how_many_zeros[0].shape

(0,)

In [13]:
# --- 6. MONTAGEM DA TABELA FINAL ---
summary_table = pd.DataFrame({
    'Absorber category': labels,
    'N(HI)_min [cm^-2]': ['0', '1.6 x 10^17', '1 x 10^19', '2 x 10^20', '1 x 10^21'],
    'N(HI)_max [cm^-2]': ['1.6 x 10^17', '1 x 10^19', '2 x 10^20', '1 x 10^21', '∞'],
    'Counts (Simulated)': category_counts.values,
    f'% of spectra in box (Simulated)': category_percentages.values
})

# Formatação estética para % (1 casa decimal)
summary_table[f'% of spectra in box (Simulated)'] = summary_table[f'% of spectra in box (Simulated)'].map('{:.1f}'.format)

print("\n" + "="*80)
print(" TABELA DE ABSORVEDORES REPRODUZIDA")
print("="*80)
display(summary_table)

# Opcional: Verificação rápida de sanidade (se a soma fecha em 100%)
soma_pct = category_percentages.sum()
print(f"\nSoma total das frações: {soma_pct:.1f}% (de {total_skewers} skewers)")


 TABELA DE ABSORVEDORES REPRODUZIDA


,Absorber category,N(HI)_min [cm^-2],N(HI)_max [cm^-2],Counts (Simulated),% of spectra in box (Simulated)
0,Lyman-α forest,0,1.6 x 10^17,2879,99.9
1,LLS,1.6 x 10^17,1 x 10^19,4,0.1
2,Sub-DLA,1 x 10^19,2 x 10^20,0,0.0
3,Small DLA,2 x 10^20,1 x 10^21,0,0.0
4,Large DLA,1 x 10^21,∞,0,0.0



Soma total das frações: 100.0% (de 2883 skewers)
